# Regularization and Hyperparameter Tuning

In this tutorial we cover **two closely related ideas** that help machine
learning models generalize to new, unseen data:

1. **Regularization** — a way to stop a model from "memorizing" the training
   data (overfitting) by adding a penalty for being too complex.
2. **Hyperparameter tuning** — a systematic way to pick the settings you choose
   *before* training (like how strong that penalty should be).

The two topics meet at the end: the strength of regularization is itself a
hyperparameter that we tune.

## Learning objectives

By the end of this notebook you should be able to:

- Recognize **overfitting** by comparing training vs. test error.
- Explain what **L1 (Lasso)** and **L2 (Ridge)** regularization do, where their
  names come from, and how they behave differently.
- Read a **regularization path** and use Lasso for **feature selection**.
- Tell the difference between a **hyperparameter** and a **learned parameter**.
- Use **cross-validation**, **`GridSearchCV`**, and a **validation curve** to
  choose good hyperparameters — for regression *and* classification — and relate
  the picture to the **bias–variance trade-off**.
- Pick a sensible **search range** for a hyperparameter, tune **two
  hyperparameters at once** (ElasticNet), and run a real **Bayesian
  optimization** loop when grid search gets too expensive.

In [ ]:
# --- All imports for this notebook live here ---
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import norm

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import (LinearRegression, Ridge, Lasso,
                                  ElasticNet, LogisticRegression)
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     validation_curve, cross_val_score)
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.datasets import load_digits, make_moons

## 1. Regularization

**Regularization** is a technique to prevent *overfitting*.

### As the word indicates, *"we want to **regularize** a model, not to overfit the data."*

**Overfitting** is the situation where a model does great on the data it was
trained on but poorly on new data it has never seen. The idea of regularization
is simple: we add a **penalty** to the loss function that grows as the model gets
more complex. The model then has to balance *fitting the data* against *staying
simple*.

### 1.1 A motivating example: overfitting

We'll create a small, noisy dataset. The **true** relationship is a simple
2nd-order polynomial:

$$ y = 0.5\,x^2 - x + 2 + \text{noise} $$

Here `x` is our input, `y` is the output, and *noise* is small random wiggle
(nothing is perfectly clean in the real world). We generate only ~30 points and
split them into a **training set** (used to fit) and a **test set** (held out to
measure generalization).

In [ ]:
# 1) Generate noisy data from a TRUE 2nd-order polynomial
def true_function(x):
    return 0.5 * x**2 - x + 2

# A LOCAL random generator with a fixed seed: re-running this cell always
# produces the exact same noise, so the whole notebook is reproducible.
rng = np.random.default_rng(1)

n_points = 30
x = np.linspace(-3, 3, n_points)
y = true_function(x) + rng.normal(0, 2.0, size=n_points)   # signal + noise

# Reshape x into a column (sklearn expects 2D feature matrices)
X = x.reshape(-1, 1)

# A dense grid of x values, used only for drawing smooth curves
x_grid = np.linspace(-3, 3, 400).reshape(-1, 1)

# 2) Split into training and test sets so we can measure generalization
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=205
)

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(x_grid, true_function(x_grid), 'g-', lw=2, label='True function')
ax.scatter(X_train, y_train, color='black', s=30, label='Training data')
ax.scatter(X_test, y_test, color='orange', s=40, marker='^', label='Test data')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('The data: a simple quadratic plus noise')
ax.legend()
plt.show()

You might think: *"If the data is complicated, can't we just use a very complex
model? Something flexible enough to fit anything?"* It's a tempting idea — and
it's exactly the trap this section is about. A model that is too flexible will
fit not only the pattern but also the **noise**, and noise doesn't repeat on new
data.

To see the trap in action, we deliberately fit an over-powered model: a
**20th-degree polynomial**. A degree-20 polynomial has 21 coefficients to play
with — vastly more flexibility than this gently-curving data actually needs.

In [ ]:
# Fit a 20th-degree polynomial with PLAIN linear regression (no regularization)
degree = 20
overfit_model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
overfit_model.fit(X_train, y_train)

# Measure error on train vs test
train_mse = mean_squared_error(y_train, overfit_model.predict(X_train))
test_mse  = mean_squared_error(y_test,  overfit_model.predict(X_test))
print(f"Degree-{degree} plain fit -> train MSE = {train_mse:.4f}   test MSE = {test_mse:,.0f}")

# Plot: true curve vs the overfit curve vs the data
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(x_grid, true_function(x_grid), 'g-', linewidth=2, label='True function')
ax.plot(x_grid, overfit_model.predict(x_grid), 'r-', linewidth=2,
        label=f'Overfit (degree {degree})')
ax.scatter(X_train, y_train, color='black', s=30, label='Training data')
ax.scatter(X_test, y_test, color='orange', s=40, marker='^', label='Test data')
ax.set_ylim(-4, 10)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'A degree-{degree} polynomial overfits ~30 noisy points')
ax.legend()
plt.show()

**What are we looking at?**

The green line is the *true* smooth curve we sampled from. The red line is our
degree-20 fit. It **wiggles violently** to pass through the black training
points — and between them it shoots right off the chart (the y-axis is clipped;
the red curve actually reaches enormous values). The model is chasing random
noise instead of the underlying pattern.

Look at the printed numbers. **MSE** (mean squared error) is the average of the
squared gaps between predictions and true values — lower is better. The
**training MSE is essentially zero** (the model threads every training point)
while the **test MSE is astronomically large**. That gap — tiny train error,
huge test error — is the signature of **overfitting**.

Regularization is one of the main tools to fix this.

### 1.2 What exactly is regularization?

Normally, linear regression finds coefficients $w$ that minimize the
**squared error** on the training data:

$$ \text{Loss} = \sum_{i} \left( y_i - \hat{y}_i \right)^2 $$

where $y_i$ is the true value, $\hat{y}_i$ is the model's prediction, and the sum
runs over all training points $i$.

Nothing in this loss says coefficients should stay small. To thread every noisy
point, our degree-20 fit used *enormous* coefficients — huge positives cancelling
huge negatives. That's the mathematical fingerprint of a wiggly, overfit curve.

Regularization changes the game by **adding a penalty term for large
coefficients** to the loss — the schematic at the top of Part 1. There are two
classic ways to measure "how large are the coefficients", and they give us the
two most famous regularizers.

Here is the whole idea in one picture:

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/regularization-overview.png" width="760" alt="regularization-overview">

The rest of Part 1 makes each panel of this picture concrete. Before we define
anything precisely, let's actually *see* overfitting happen.

#### L1 regularization — **Lasso** regression

$$ \text{Loss} = \sum_i (y_i - \hat{y}_i)^2 \;+\; \alpha \sum_j |w_j| $$

- $w_j$ is the $j$-th **coefficient** (how much feature $j$ influences the prediction).
- $\sum_j |w_j|$ is the sum of the coefficients' **absolute values** — the "L1" penalty.
- $\alpha$ (alpha) is a knob **you** set: bigger $\alpha$ = stronger pressure to
  keep coefficients small.

**Why "Lasso"?** It's an acronym (Tibshirani, 1996): **L**east **A**bsolute
**S**hrinkage and **S**election **O**perator. The "selection" part is the key.

**How it behaves:** the absolute value $|w|$ has a sharp **corner at zero** — the
penalty pushes with the *same* force no matter how small the coefficient already
is. Once a feature isn't pulling its weight, that constant push shoves its
coefficient **exactly to zero**, deleting the feature from the model. Lasso
therefore does **automatic feature selection**.

*Geometric intuition (optional):* you can picture the penalty as a "budget
region" the coefficients must stay inside — for L1 that region is a **diamond**.
The best fit tends to land where the region touches the error surface, and a
diamond's sharp **corners sit exactly on the axes** — where some coefficients
are zero.

#### L2 regularization — **Ridge** regression

$$ \text{Loss} = \sum_i (y_i - \hat{y}_i)^2 \;+\; \alpha \sum_j w_j^2 $$

The other classic penalty **squares** the coefficients instead of taking absolute
values — the "L2" penalty. Same $\alpha$ knob, same goal, different personality.

**Why "Ridge"?** The name comes from the closed-form solution (Hoerl & Kennard,
1970): the penalty adds $\alpha$ along the diagonal of the matrix
$X^\top X$ — literally a *ridge* running down the diagonal — which also makes the
math more stable.

**How it behaves:** squaring means the penalty gets *very* angry about big
coefficients but barely notices tiny ones — the push fades as a coefficient
approaches zero. So Ridge **shrinks all coefficients smoothly toward zero, but
almost never exactly to zero**. In the budget-region picture, L2's region is a
**circle**: no corners, so solutions rarely land exactly on an axis. Ridge is a
great default when you believe most features matter at least a little.

#### Ridge vs. Lasso at a glance

| | **L1 / Lasso** | **L2 / Ridge** |
|---|---|---|
| Penalty | sum of absolute values $\sum \lvert w_j\rvert$ | sum of squares $\sum w_j^2$ |
| Effect on coefficients | can push some **exactly to 0** | shrinks them all toward 0 (rarely exactly 0) |
| Side benefit | **feature selection** (drops useless features) | smoother, stable fits |
| Name origin | acronym: Least Absolute Shrinkage and Selection Operator | "ridge" added to the diagonal of $X^\top X$ |

Because huge, wiggly fits need large coefficients, *either* penalty makes the
fitted curve smoother — exactly what we want against overfitting. They just get
there differently.

### 1.3 Watching `alpha` control the fit

Now let's fit **Ridge** and **Lasso** on the *same* degree-20 polynomial
features, but sweep the penalty strength `alpha` across four values:
`0` (no penalty at all), `0.001` (tiny), `0.1` (moderate), and `1000` (very
strong).

We wrap everything in a **`Pipeline`** with `StandardScaler`. Scaling puts all
the polynomial features on a comparable range, which keeps the math numerically
stable and makes the penalty affect features fairly.

In [ ]:
alphas = [0, 0.001, 0.1, 1000]

def poly_model(estimator):
    # polynomial features -> StandardScaler -> regularized linear model
    return make_pipeline(PolynomialFeatures(degree),
                         StandardScaler(),
                         estimator)

fig, axes = plt.subplots(2, len(alphas), figsize=(16, 7), dpi=300,
                         sharex=True, sharey=True)

for col, a in enumerate(alphas):
    # Ridge on top row, Lasso on bottom row.
    # alpha=0 means "no regularization" -> use plain LinearRegression for both.
    ridge = LinearRegression() if a == 0 else Ridge(alpha=a)
    lasso = LinearRegression() if a == 0 else Lasso(alpha=a, max_iter=200000)

    for row, (name, est) in enumerate([('Ridge (L2)', ridge), ('Lasso (L1)', lasso)]):
        model = poly_model(est)
        model.fit(X_train, y_train)
        te = mean_squared_error(y_test, model.predict(X_test))

        ax = axes[row, col]
        ax.plot(x_grid, true_function(x_grid), 'g-', lw=1.5)
        ax.plot(x_grid, model.predict(x_grid), 'r-', lw=2)
        ax.scatter(X_train, y_train, color='black', s=12)
        ax.set_ylim(-4, 10)
        ax.set_title(f'{name}\nalpha={a} | test MSE={te:,.2f}', fontsize=10)

axes[0, 0].set_ylabel('Ridge')
axes[1, 0].set_ylabel('Lasso')
fig.suptitle(f'Same degree-{degree} features, different regularization strength (alpha)',
             fontsize=13)
plt.tight_layout()
plt.show()

**Reading the grid (left to right = weaker to stronger penalty):**

- **`alpha = 0`** (far left): no penalty. The curve explodes off the chart and the
  test MSE is in the hundreds of thousands — classic overfitting.
- **tiny `alpha` (0.001)**: even a whisper of a penalty tames the explosion,
  though the curve still wiggles noticeably.
- **moderate `alpha` (0.1)**: the sweet spot in this sweep — the fit hugs the
  green curve and the test MSE is the lowest in **both** rows.
- **`alpha = 1000`** (far right): the penalty is *too* strong. Ridge is squashed
  nearly flat; Lasso has zeroed out **every** coefficient, so it predicts a
  constant (the training-set average). Both test MSEs **rise again** — the models
  are now **underfitting**.

So `alpha` trades off overfitting (too wiggly) against underfitting (too flat),
and the test MSE is lowest at a **middle** value. Finding that middle value
automatically is exactly what hyperparameter tuning will do in Part 2.

#### Coefficients: L1 zeros them out, L2 just shrinks them

Let's print the coefficient magnitudes for a moderate `alpha` to *see* the
difference between L1 and L2 that we described in the table. With degree 20
there are **21 coefficients** (powers $x^0$ through $x^{20}$).

In [ ]:
a = 0.1
ridge_model = poly_model(Ridge(alpha=a)).fit(X_train, y_train)
lasso_model = poly_model(Lasso(alpha=a, max_iter=200000)).fit(X_train, y_train)

# The linear model is the LAST step of each pipeline.
ridge_coefs = ridge_model[-1].coef_
lasso_coefs = lasso_model[-1].coef_

print(f"Number of coefficients: {len(ridge_coefs)}")
print(f"Ridge: coefficients exactly 0 -> {np.sum(np.isclose(ridge_coefs, 0))}")
print(f"Lasso: coefficients exactly 0 -> {np.sum(np.isclose(lasso_coefs, 0))}")
print(f"Lasso kept only these powers of x: {np.where(~np.isclose(lasso_coefs, 0))[0].tolist()}")

idx = np.arange(len(ridge_coefs))
width = 0.4
fig, ax = plt.subplots(figsize=(11, 5), dpi=300)
ax.bar(idx - width/2, np.abs(ridge_coefs), width, label='Ridge (L2)')
ax.bar(idx + width/2, np.abs(lasso_coefs), width, label='Lasso (L1)')
ax.set_xlabel('Polynomial feature index (power of x)')
ax.set_ylabel('|coefficient|')
ax.set_xticks(idx)
ax.set_title(f'Coefficient magnitudes at alpha={a}')
ax.legend()
plt.show()

**Takeaway:** with the same `alpha`, **Lasso (L1) sets most of the 21
coefficients to exactly 0** and keeps only the four lowest powers — it literally
throws away features it finds unhelpful (this is *feature selection*).
**Ridge (L2) keeps essentially all of them but shrinks them** toward zero. Both
make the fit smoother; they just get there differently.

### 1.4 The regularization path: every coefficient vs. `alpha`

So far we looked at snapshots at a few `alpha` values. A **regularization path**
shows the whole movie: we sweep `alpha` continuously and plot **how every
coefficient changes**. Each colored line below is one coefficient (one power of
$x$) as `alpha` grows from tiny to huge.

In [ ]:
# At near-zero alpha, Lasso's optimizer warns that it stopped early.
# That is harmless for this plot, so we silence just that warning.
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

path_alphas = np.logspace(-4, 3, 60)

ridge_paths = []
lasso_paths = []
for a in path_alphas:
    ridge_paths.append(poly_model(Ridge(alpha=a)).fit(X_train, y_train)[-1].coef_)
    lasso_paths.append(poly_model(Lasso(alpha=a, max_iter=200000)).fit(X_train, y_train)[-1].coef_)
ridge_paths = np.array(ridge_paths)   # shape (60, 21): one row per alpha
lasso_paths = np.array(lasso_paths)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300, sharey=True)
for ax, paths, name in [(axes[0], ridge_paths, 'Ridge (L2)'),
                        (axes[1], lasso_paths, 'Lasso (L1)')]:
    ax.plot(path_alphas, paths)
    ax.set_xscale('log')
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_xlabel('alpha (log scale)')
    ax.set_title(f'{name}: coefficient paths')
    # ax.set_yscale('log')
axes[0].set_ylabel('coefficient value')
fig.suptitle('The regularization path: each line is one coefficient', fontsize=13)
plt.tight_layout()
plt.show()

**Reading the paths:** on the left (tiny `alpha`) both models use large
coefficients. As `alpha` grows:

- **Ridge** lines glide **smoothly toward zero** but stay nonzero for a long time
  — every feature keeps a (shrinking) say.
- **Lasso** lines **hit exactly zero one after another** and stay there — features
  are eliminated one by one, strongest survivors last.

This is the table from section 1.2, drawn as a single picture.

### 1.5 Lasso as an automatic feature detector

Here's a fun consequence. Our data really came from a **quadratic**:
$y = 0.5x^2 - x + 2$. Out of the 21 polynomial features we offered
($x^0 \dots x^{20}$), only $x^1$ and $x^2$ actually matter. Can Lasso *figure
that out on its own*, just from the noisy data? Let's watch which powers survive
as we crank up `alpha`.

In [ ]:
for a in [0.01, 0.1, 1, 3]:
    m = poly_model(Lasso(alpha=a, max_iter=200000)).fit(X_train, y_train)
    surviving = np.where(~np.isclose(m[-1].coef_, 0))[0]
    print(f"alpha = {a:<5} -> surviving powers of x: {surviving.tolist()}")

**Takeaway:** as the penalty grows, Lasso strips the model down — from a handful
of features, to just the **low powers** ($x^1 \dots x^4$) that describe a gentle
curve, and finally (at `alpha = 3`) to *nothing at all* (an empty model that just
predicts the average — the flat line we saw in section 1.3).

Two honest footnotes worth learning from:

- Lasso doesn't recover *exactly* $\{x^1, x^2\}$. On our range of $x$, the
  scaled features $x^2$ and $x^4$ look **very similar** (they are strongly
  *correlated*), so Lasso sometimes keeps $x^4$ as a stand-in for $x^2$. **With
  correlated features, *which* one survives is somewhat arbitrary** — remember
  this; it becomes an argument for ElasticNet in section 2.6.
- Still, the big story holds: out of 21 candidate features, Lasso automatically
  zeroed in on the few low powers that matter. This is why L1 regularization is
  so popular when you have many candidate features and suspect only a few are
  real.

## 2. Hyperparameter Tuning

We just saw that `alpha` matters a lot — but how do we *choose* a good value
instead of guessing? That is the job of **hyperparameter tuning**.

### 2.1 Hyperparameter vs. learned parameter

These two words sound similar but mean different things:

- A **learned parameter** is something the model *figures out from the data*
  during training. Example: the coefficients $w_j$ in our regression. You never
  set these by hand.
- A **hyperparameter** is a setting *you choose before training starts*. The
  model does **not** learn it from the data. Example: `alpha` in Ridge/Lasso.

More examples of hyperparameters you'll meet:
- the **learning rate** in gradient descent,
- the **max depth** of a decision tree,
- the number of trees (`n_estimators`) in a random forest,
- **k**, the number of clusters in k-means.

Tuning these well can dramatically improve a model.

For example, This is how we update our learned parameters using gradient descent. 

$$ w \leftarrow w - \gamma\,\frac{\partial \text{Loss}}{\partial w} = w - \gamma\,dw
\qquad\qquad
b \leftarrow b - \gamma\,\frac{\partial \text{Loss}}{\partial b} = b - \gamma\,db $$

$dw$ and $db$ are exactly what the `gradients` function above computes. One pass of this update over the whole training set is an **epoch**; repeating it for many epochs shrinks the loss step by step.

### This suggests that we have derivative of w and b, but not coefficient of regularization term ($\alpha$).

#### So we need to tune our model and hyperparameters, outside of gradient descent. 

### 2.2 Train / validation / test, and cross-validation

So far we have used the two-way split from Part 1:

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/train-test-split.png" width="640" alt="train-test-split">

**With only these two sets, we cannot do hyperparameter tuning.** To compare
different `alpha` values we'd have to score them on *some* held-out data — and
the only held-out data here is the test set. But picking `alpha` by test-set
score is **cheating**: the test set is supposed to be a final, untouched exam,
and once we've used it to make choices, it can no longer tell us honestly how
the model performs on unseen data.

So in principle we split **three ways**, each part with its own job:

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/train-val-test-split.png" width="640" alt="train-val-test-split">

But carving out a single fixed validation set wastes data and can be noisy —
maybe the few points that landed in it just happen to be unusual.
**k-fold cross-validation (CV)** fixes this: split the *training* data into *k*
equal parts ("folds"). Train on *k−1* of them, validate on the one left out, and
repeat so every fold gets a turn as the validation set. Average the *k* scores.

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/kfold-cv.png" width="640" alt="kfold-cv">

`GridSearchCV` in scikit-learn does all of this automatically: you give it a list
of hyperparameter values to try, and it runs cross-validation for each one and
reports the best.

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/grid-search.png" width="760" alt="grid-search">

### 2.3 Tuning example on linear regression (Ridge `alpha`)

Let's tune the regularization strength `alpha` for **Ridge** on the same
polynomial data from Part 1. We try many `alpha` values spaced evenly on a
**log scale** (so we cover very small to fairly large values), and let
`GridSearchCV` pick the best by 5-fold cross-validation.

In [ ]:
# Log-spaced alphas from 1e-4 up to 1e3
alpha_grid = np.logspace(-4, 3, 30)

ridge_pipe = make_pipeline(PolynomialFeatures(degree), StandardScaler(), Ridge())

# The parameter name follows the "step name + __ + parameter" convention.
param_grid = {'ridge__alpha': alpha_grid}

grid = GridSearchCV(
    ridge_pipe, param_grid, cv=5,
    scoring='neg_mean_squared_error'  # sklearn maximizes score, so it uses NEGATIVE MSE
)
grid.fit(X_train, y_train)

best_alpha = grid.best_params_['ridge__alpha']
print(f"Best alpha found by cross-validation: {best_alpha:.4g}")

# Turn negative-MSE back into plain MSE for a readable plot (lower = better).
mean_cv_mse = -grid.cv_results_['mean_test_score']

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(alpha_grid, mean_cv_mse, 'o-')
ax.axvline(best_alpha, color='red', linestyle='--', label=f'best alpha = {best_alpha:.3g}')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('alpha (log scale)')
ax.set_ylabel('Mean cross-validation MSE (log scale)')
ax.set_title('Choosing alpha for Ridge by 5-fold cross-validation')
ax.legend()
plt.show()

In [ ]:
# Fit the final model with the best alpha and see the curve.
best_ridge = grid.best_estimator_
test_mse_best = mean_squared_error(y_test, best_ridge.predict(X_test))
print(f"Test MSE with tuned alpha = {test_mse_best:.3f}"
      f"   (plain degree-{degree} fit was {test_mse:,.0f})")

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(x_grid, true_function(x_grid), 'g-', lw=2, label='True function')
ax.plot(x_grid, best_ridge.predict(x_grid), 'b-', lw=2,
        label=f'Tuned Ridge (alpha={best_alpha:.3g})')
ax.scatter(X_train, y_train, color='black', s=30, label='Training data')
ax.set_ylim(-4, 10)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Final fit using the cross-validated best alpha')
ax.legend()
plt.show()

**Takeaway:** the U-shaped curve shows the classic tradeoff — too-small `alpha`
(left) overfits and too-large `alpha` (right) underfits, with a sweet spot in the
middle. `GridSearchCV` found that sweet spot for us, and the resulting blue curve
tracks the true green curve far better than the exploding red curve from Part 1 —
with a vastly lower test MSE, and *without ever peeking at the test set to choose
`alpha`*.

### 2.4 The validation curve: seeing overfitting and underfitting at once

The plot above showed only the cross-validation error. A **validation curve**
adds the **training error** to the same picture, and the *gap* between the two
lines is where the story is:

- **big gap** (train great, validation bad) → overfitting,
- **no gap but both bad** → underfitting.

scikit-learn's `validation_curve` computes both for a sweep of one
hyperparameter.

In [ ]:
train_scores, cv_scores = validation_curve(
    ridge_pipe, X_train, y_train,
    param_name='ridge__alpha', param_range=alpha_grid,
    cv=5, scoring='neg_mean_squared_error'
)

# Average over the 5 folds, flip negative-MSE back to plain MSE.
train_mse_curve = -train_scores.mean(axis=1)
cv_mse_curve = -cv_scores.mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(alpha_grid, train_mse_curve, 'o-', label='Training MSE')
ax.plot(alpha_grid, cv_mse_curve, 's-', label='Cross-validation MSE')
ax.axvline(best_alpha, color='red', linestyle='--', label=f'best alpha = {best_alpha:.3g}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('alpha (log scale)')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Validation curve for Ridge alpha')
ax.legend()
plt.show()

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/train-val-test-split.png" width="640" alt="train-val-test-split">


**Reading the validation curve:**

- **Left side (tiny `alpha`)**: training MSE is much lower than CV MSE — the
  model memorizes the training folds but generalizes poorly. **Overfitting.**
- **Right side (huge `alpha`)**: the two lines meet, but *both* rise — the model
  is too constrained to fit even the training data. **Underfitting.**
- **The sweet spot** is where the CV line bottoms out — the same `alpha`
  GridSearchCV picked.

This train-vs-validation picture is one of the most useful diagnostic plots in
all of machine learning; you will draw it for many models beyond Ridge.

#### Is this the bias–variance trade-off? Yes!

What you just saw has a famous name. A model's error on new data can be split
into two ingredients:

- **Bias**: error from being **too rigid** — the model *systematically* misses
  the pattern, and would keep missing it the same way no matter which training
  sample it saw. Our nearly-flat, huge-`alpha` fits have **high bias**
  (= underfitting).
- **Variance**: error from being **too sensitive** — redraw the noisy training
  points and the fitted curve changes wildly, because the model chases each
  sample's noise. Our exploding, `alpha≈0` fits have **high variance**
  (= overfitting).

Shrinking coefficients makes the model less sensitive to the training sample, so
**increasing `alpha` trades variance for bias**: the left side of the validation
curve is the high-variance regime, the right side is the high-bias regime, and
the sweet spot in the middle is the best *balance* of the two — which is why this
is called the **bias–variance trade-off**. Every hyperparameter that controls
model complexity (`alpha`, `C`, `max_depth`, polynomial degree…) slides the model
along this same trade-off.

### 2.5 Choosing the search *range* matters

`GridSearchCV` can only pick the best value **among the ones you offer**. If your
grid covers the wrong range, you get a confident-looking — but bad — answer.
Let's tune the same Ridge model with a **badly chosen range**: five alphas
between $10^{-6}$ and $10^{-4}$, all far too small.

In [ ]:
bad_grid = np.logspace(-6, -4, 5)     # all of these are ~zero penalty!
grid_bad = GridSearchCV(ridge_pipe, {'ridge__alpha': bad_grid},
                        cv=5, scoring='neg_mean_squared_error')
grid_bad.fit(X_train, y_train)

bad_best = grid_bad.best_params_['ridge__alpha']
bad_test_mse = mean_squared_error(y_test, grid_bad.best_estimator_.predict(X_test))

print(f"Bad grid searched:  {bad_grid}")
print(f"'Best' alpha found: {bad_best:.1e}   <- sits on the EDGE of the grid!")
print(f"Test MSE with it:   {bad_test_mse:.2f}")
print(f"(For comparison, the well-ranged search in 2.3 reached test MSE {test_mse_best:.2f})")

fig, ax = plt.subplots(figsize=(8, 4), dpi=300)
ax.plot(bad_grid, -grid_bad.cv_results_['mean_test_score'], 'o-')
ax.axvline(bad_best, color='red', linestyle='--', label='best (on the edge!)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('alpha (log scale)'); ax.set_ylabel('Mean CV MSE (log scale)')
ax.set_title('A badly-ranged grid: the CV error is still falling at the edge')
ax.legend()
plt.show()

**What went wrong, and the rules of thumb:**

The search dutifully returned a "best" alpha — but its test MSE is several times
worse than the well-ranged search from 2.3. The tell-tale sign is in the plot:
the CV error is **still decreasing at the boundary of the grid**, so the true
optimum lies *outside* the range we offered.

1. **If the best value lands on the edge of your grid, widen the grid** and
   search again. An interior minimum is what a trustworthy result looks like.
2. For scale-type hyperparameters like `alpha` (or learning rates), space the
   grid on a **log scale** — the difference between 0.001 and 0.01 matters as
   much as between 1 and 10.
3. Go **coarse to fine**: first a wide, sparse grid to find the right
   neighborhood, then a narrower, denser grid around the winner. (Section 2.9
   shows how Bayesian optimization automates this instinct.)

### 2.6 Tuning two hyperparameters at once: ElasticNet (L1 + L2 together)

Why choose between Ridge and Lasso? **ElasticNet** uses *both* penalties at once:

$$ \text{Loss} = \sum_i (y_i - \hat{y}_i)^2
   \;+\; \alpha \Big( \rho \sum_j |w_j| \;+\; \tfrac{1-\rho}{2} \sum_j w_j^2 \Big) $$

- $\alpha$ still controls the **overall strength** of regularization,
- $\rho$ (`l1_ratio` in sklearn) controls the **mix**: $\rho = 1$ is pure Lasso,
  $\rho = 0$ is pure Ridge, values in between blend the two.

That gives us **two hyperparameters to tune together**. `GridSearchCV` handles
this the obvious way: try **every combination** (here 9 alphas × 5 ratios = 45
fits, each cross-validated 5 times). We can then draw the results as a
**heatmap**. Because the CV errors span several orders of magnitude, we put the
**colors on a log scale** too — otherwise the whole "good" region would collapse
into one indistinguishable shade — and print the value in each cell.

In [ ]:
en_pipe = make_pipeline(PolynomialFeatures(degree), StandardScaler(),
                        ElasticNet(max_iter=100000))

en_alphas = np.logspace(-3, 1, 9)
en_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
en_param_grid = {'elasticnet__alpha': en_alphas,
                 'elasticnet__l1_ratio': en_ratios}

grid_en = GridSearchCV(en_pipe, en_param_grid, cv=5,
                       scoring='neg_mean_squared_error')
grid_en.fit(X_train, y_train)

print("Best combination:", grid_en.best_params_)
en_test_mse = mean_squared_error(y_test, grid_en.best_estimator_.predict(X_test))
print(f"ElasticNet test MSE = {en_test_mse:.3f}   (tuned Ridge alone was {test_mse_best:.3f})")

# Reshape the CV results into an (n_alphas x n_ratios) table for the heatmap.
cv_mse_table = -grid_en.cv_results_['mean_test_score'].reshape(len(en_alphas),
                                                               len(en_ratios))

fig, ax = plt.subplots(figsize=(8, 6.5), dpi=300)
im = ax.imshow(cv_mse_table, cmap='viridis_r', aspect='auto',
               norm=LogNorm())          # log colors: small differences near the
fig.colorbar(im, ax=ax,                 # minimum stay visible
             label='Mean CV MSE (log color scale, lower = better)')

# Print the CV MSE inside every cell, and outline the best one in red.
best_i, best_j = np.unravel_index(np.argmin(cv_mse_table), cv_mse_table.shape)
for i in range(len(en_alphas)):
    for j in range(len(en_ratios)):
        ax.text(j, i, f'{cv_mse_table[i, j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if cv_mse_table[i, j] > 2 else 'black')
ax.add_patch(plt.Rectangle((best_j - 0.5, best_i - 0.5), 1, 1,
                           fill=False, edgecolor='red', lw=2.5))

ax.set_xticks(range(len(en_ratios)), en_ratios)
ax.set_yticks(range(len(en_alphas)), [f'{a:.3g}' for a in en_alphas])
ax.set_xlabel('l1_ratio  (0 = pure Ridge ... 1 = pure Lasso)')
ax.set_ylabel('alpha')
ax.set_title('ElasticNet: CV error over the 2-D hyperparameter grid\n(red box = best)')
plt.show()

**Takeaway:** the heatmap shows the whole 2-D landscape at a glance — each cell
is one (alpha, l1_ratio) combination, brighter = lower CV error, and the red box
marks the winner. Here the best mix is a **balanced blend** of the two penalties
— and it beats our tuned pure-Ridge model on the test set. Section 1.5 hints at
why a blend helps: our polynomial features are strongly **correlated**, and pure
L1 picks one member of a correlated group somewhat arbitrarily, while the L2
ingredient spreads the weight more stably across them. When you don't know in
advance which penalty suits your data, ElasticNet lets cross-validation decide
the mix for you.

Note the cost, though: **grids grow multiplicatively**. Two hyperparameters with
9 and 5 values already mean 45 combinations; add a third and you're in the
hundreds. Keep that in mind for section 2.9.

### 2.7 Regularization in classification: logistic regression and `C`

Regularization is not just for regression. **Logistic regression** (see the
`logistic-regression-demo` notebook in this folder) is a *classifier*, and in
scikit-learn it is regularized by default. One catch:

> its knob is called **`C`, and it is the *inverse* of the penalty strength:**
> $C \approx 1/\alpha$. **Small `C` = strong regularization**, large `C` = weak.

We'll use the classic `make_moons` toy dataset — two interleaving crescents —
and give logistic regression degree-20 polynomial features so it *can* draw a
curvy boundary. The question is how curvy it *should* be, and `C` decides. The
top row below shows the decision boundary for three values of `C`; the bottom
row shows each model's train vs. test accuracy as bars.

In [ ]:
# Two noisy interleaving half-moons: a classic 2-D classification toy dataset.
X_moons, y_moons = make_moons(n_samples=100, noise=0.2, random_state=1)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=205
)

# A grid of points covering the plane, to draw each model's decision regions.
xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.8, 2.3, 300))
plane = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), dpi=300)
for col, C in enumerate([1e-3, 1, 1e+3]):
    clf = make_pipeline(PolynomialFeatures(30), StandardScaler(),
                        LogisticRegression(C=C, max_iter=20000))
    clf.fit(Xm_train, ym_train)
    train_acc = clf.score(Xm_train, ym_train)
    test_acc = clf.score(Xm_test, ym_test)

    # Top row: decision boundary
    ax = axes[0, col]
    zz = clf.predict(plane).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
    ax.scatter(Xm_train[:, 0], Xm_train[:, 1], c=ym_train, cmap='coolwarm',
               s=25, edgecolors='k')
    ax.set_title(f'C={C}')

    # Bottom row: train vs test accuracy for the same model
    ax = axes[1, col]
    bars = ax.bar(['train', 'test'], [train_acc, test_acc],
                  color=['steelblue', 'darkorange'])
    ax.bar_label(bars, fmt='%.2f')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('accuracy')
    ax.set_title(f'C={C}: train vs test accuracy')

fig.suptitle('Same model, three C values: strong -> moderate -> weak regularization',
             fontsize=13)
plt.tight_layout()
plt.show()

**Reading the columns:**

- **`C = 0.001`** (strong regularization): the boundary is so constrained it
  barely bends — it can't follow the moons at all. Train *and* test accuracy are
  both poor: **underfitting** (high bias).
- **`C = 1`** (moderate): a smooth curve that follows the two crescents, and the
  best test accuracy of the three.
- **`C = 1000`** (weak regularization): the boundary contorts itself around
  individual noisy points. Training accuracy hits 100%, but test accuracy
  *drops*: **overfitting** (high variance) — the same signature as our wiggly
  polynomial, now visible directly in the bar plots.

(A caution when reading the bars: the test set here has only 20 points, so each
single point is worth 5% of accuracy — small differences are noisy.)

And of course, `C` is a hyperparameter — so we tune it the standard way:

In [ ]:
grid_C = GridSearchCV(
    make_pipeline(PolynomialFeatures(30), StandardScaler(),
                  LogisticRegression(max_iter=20000)),
    {'logisticregression__C': np.logspace(-3, 3, 50)},
    cv=5   # default scoring for a classifier: accuracy
)
grid_C.fit(Xm_train, ym_train)

print(f"Best C: {grid_C.best_params_['logisticregression__C']:.4g}")
print(f"Test accuracy with tuned C: {grid_C.score(Xm_test, ym_test):.3f}")

Same recipe as always: log-spaced grid, 5-fold CV, never touching the test set
until the end. The only things that changed are the *scoring* — for classifiers
`GridSearchCV` defaults to **accuracy** (higher = better) instead of MSE — and
the direction of the knob (large `C` = *weak* penalty). Cross-validation picks a
`C` in the moderate zone, using the training data alone.

### 2.8 Tuning example on a bigger model (Random Forest)

The same recipe works for any model — including ones with no coefficients at
all. Here we tune a **Random Forest classifier** on the `digits` dataset —
8×8 grayscale images of handwritten digits 0–9 that ship with scikit-learn
(small and fast, but still real image data).

A **random forest** builds many decision trees and lets them vote. Two important
hyperparameters:
- `n_estimators`: how many trees are in the forest.
- `max_depth`: how deep each tree may grow (deeper = more complex).

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/random-forest-hyperparameters.png" width="760" alt="random-forest-hyperparameters">

In [ ]:
# Load the small handwritten-digits dataset (1797 images, 8x8 pixels each)
digits = load_digits()
all_images = digits.images     # shape (1797, 8, 8)
all_labels = digits.target     # the digit 0-9 in each image

print("images shape:", all_images.shape, "| labels shape:", all_labels.shape)

# Peek at a few examples
fig, axes = plt.subplots(1, 6, figsize=(9, 2), dpi=300)
for ax, img, lab in zip(axes, all_images, all_labels):
    ax.imshow(img, cmap='gray'); ax.set_title(str(lab)); ax.axis('off')
plt.show()

# Split into training and test sets
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    all_images, all_labels, test_size=0.2, random_state=205)

We define `param_grid`, a dictionary listing the hyperparameter values to try.
`GridSearchCV` trains a random forest for **every combination** (3 × 3 = 9 here)
and uses **5-fold cross-validation** to score each one, keeping the best.

Note `X_train_d.reshape(len(X_train_d), -1)`: each 8×8 image is **flattened** into
a row of 64 numbers, because the classifier expects each sample to be a flat vector
of features rather than a 2D grid.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],   # number of trees
    'max_depth': [None, 10, 20],      # how deep each tree may grow (None = unlimited)
}

rf_model = RandomForestClassifier(random_state=205)
grid_search = GridSearchCV(rf_model, param_grid, cv=5)

# Flatten each 8x8 image into a 64-length vector before fitting.
grid_search.fit(X_train_d.reshape(len(X_train_d), -1), y_train_d)

print("Best hyperparameters:", grid_search.best_params_)

#### Visualizing the tuning results

We pull the mean cross-validation score for every combination out of
`grid_search.cv_results_` and draw a scatter plot: `n_estimators` on the x-axis,
`max_depth` on the y-axis, and **color = mean CV accuracy**. Brighter points did
better. (`max_depth=None` means unlimited depth; we plot it as a large number so
it fits on the axis.)

In [ ]:
mean_scores = grid_search.cv_results_['mean_test_score']
param_combinations = list(grid_search.cv_results_['params'])

for score, params in zip(mean_scores, param_combinations):
    print(f"Mean CV accuracy: {score:.4f}  |  {params}")

n_estimators_values = [p['n_estimators'] for p in param_combinations]
# Replace None (unlimited depth) with a big number just for plotting.
max_depth_values = [(p['max_depth'] if p['max_depth'] is not None else 30)
                    for p in param_combinations]

fig, ax = plt.subplots(figsize=(9, 6), dpi=300)
sc = ax.scatter(n_estimators_values, max_depth_values, c=mean_scores,
                cmap='viridis', s=250, edgecolors='k')
fig.colorbar(sc, ax=ax, label='Mean CV accuracy')
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth (None shown as 30)')
ax.set_title('Hyperparameter tuning results (brighter = better)')
plt.show()

#### The best model

After the search, `grid_search.best_estimator_` is the random forest that was
re-trained on all the training data using the winning hyperparameters. We can use
it directly to make predictions.

In [ ]:
best_model = grid_search.best_estimator_
test_acc = best_model.score(X_test_d.reshape(len(X_test_d), -1), y_test_d)
print("Best model:", best_model)
print(f"Accuracy on the held-out test set: {test_acc:.3f}")

#### Tying it together: `max_depth` *is* regularization

Here is the connection between our two topics. Limiting a tree's `max_depth` is
itself a form of **regularization**: a shallow tree cannot carve the data into
tiny memorized pieces, so it is forced to stay simpler and generalize better —
exactly the role `alpha` played for Ridge and Lasso, and `1/C` for logistic
regression.

Below we train a random forest with a deliberately small `max_depth`. The depth
limit keeps each tree simple and helps prevent overfitting, just like a penalty
term does in linear models.

In [ ]:
regularized_model = RandomForestClassifier(max_depth=5, random_state=205)
regularized_model.fit(X_train_d.reshape(len(X_train_d), -1), y_train_d)

reg_acc = regularized_model.score(X_test_d.reshape(len(X_test_d), -1), y_test_d)
print(f"Shallow forest (max_depth=5) test accuracy: {reg_acc:.3f}")
print("A depth limit acts like a regularizer: it keeps each tree simple.")

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/grid-search.png" width="760" alt="grid-search">

### 2.9 Beyond grid search: random search and Bayesian optimization

Grid search is simple and thorough — but it **scales terribly**. Every extra
hyperparameter *multiplies* the number of combinations: 5 hyperparameters with 10
values each is $10^5 = 100{,}000$ model fits, each cross-validated. Two smarter
families of methods exist:

- **Random search**: instead of a rigid grid, sample combinations *at random*
  from ranges you specify, and stop after a fixed budget (say, 50 tries). It
  sounds too simple to work, but it usually beats a grid of the same budget:
  a grid wastes many fits re-testing the same few values of an unimportant
  hyperparameter, while random sampling explores far more distinct values of
  each one. scikit-learn ships this as `RandomizedSearchCV`.

- **Bayesian optimization**: the smartest of the three, and the one we'll build
  for real. Grid and random search are *blind* — try number 40 ignores
  everything learned from tries 1–39. Bayesian optimization instead keeps two
  objects:

  1. a **surrogate model** — a cheap statistical model (typically a **Gaussian
     process**) fitted to the results so far, which predicts, for *any*
     candidate hyperparameter, both an expected score **and how uncertain that
     prediction is**;
  2. an **acquisition function** — a rule that scores each candidate by how
     *useful* trying it would be. We'll use **Expected Improvement (EI)**: how
     much do we expect this candidate to beat the best result so far? EI is
     large where the surrogate predicts a good score (**exploitation**) *and*
     where the surrogate is very unsure (**exploration**), so the search
     balances the two automatically.

  Loop: fit surrogate → maximize acquisition → evaluate that candidate → repeat.
  Each expensive evaluation is chosen using *all* the information gathered so
  far, so strong settings are found in far fewer tries — the thing that matters
  when one training run takes hours.

Everything we need for a real (if minimal) Bayesian optimizer ships with
scikit-learn and scipy: `GaussianProcessRegressor` is the surrogate, and EI is
four lines of formulas. Let's tune Ridge's `alpha` with it.

The idea, iteration by iteration — the dashed curve is the true (unknown) objective we are minimizing; we can only afford a few evaluations of it:

<img src="https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/main/02-supervised-learning/regression-and-regularization-and-hyperparameter-tuning/images/bayesian-optimization.png" width="660" alt="bayesian-optimization">

In [ ]:
# --- A minimal but REAL Bayesian optimization loop for Ridge alpha ---

def cv_mse(log_alpha):
    """The expensive black-box function we are optimizing:
    5-fold CV error of Ridge at alpha = 10**log_alpha."""
    model = make_pipeline(PolynomialFeatures(degree), StandardScaler(),
                          Ridge(alpha=10.0 ** log_alpha))
    scores = cross_val_score(model, X_train, y_train, cv=5,
                             scoring='neg_mean_squared_error')
    return -scores.mean()

# We search log10(alpha) in [-4, 3] -- the same range as section 2.3.
# Start with 4 evenly spread "starter" evaluations.
tried_log_alpha = list(np.linspace(-4, 3, 4))
tried_mse = [cv_mse(la) for la in tried_log_alpha]

candidates = np.linspace(-4, 3, 300)          # where the acquisition is evaluated
gp = GaussianProcessRegressor(kernel=Matern(nu=2.5),  # the surrogate model
                              normalize_y=True, alpha=1e-6, random_state=0)

for it in range(1, 9):
    # 1) Fit the surrogate to everything tried so far.
    #    (We model log10(MSE): it varies smoothly across our range.)
    gp.fit(np.array(tried_log_alpha).reshape(-1, 1), np.log10(tried_mse))

    # 2) Surrogate's prediction and uncertainty at every candidate.
    mu, sigma = gp.predict(candidates.reshape(-1, 1), return_std=True)

    # 3) Expected Improvement over the best value seen so far (we minimize).
    best_so_far = np.log10(min(tried_mse))
    improvement = best_so_far - mu
    z = np.where(sigma > 0, improvement / sigma, 0.0)
    ei = improvement * norm.cdf(z) + sigma * norm.pdf(z)

    # 4) Evaluate the candidate with the highest EI, add it to the record.
    next_la = float(candidates[np.argmax(ei)])
    tried_log_alpha.append(next_la)
    tried_mse.append(cv_mse(next_la))
    print(f"Iter {it}: EI chose alpha = {10**next_la:9.4g}"
          f"  ->  CV MSE = {tried_mse[-1]:6.3f}"
          f"   (best so far: {min(tried_mse):.3f})")

bo_best_idx = int(np.argmin(tried_mse))
bo_alpha = 10 ** tried_log_alpha[bo_best_idx]
bo_model = make_pipeline(PolynomialFeatures(degree), StandardScaler(),
                         Ridge(alpha=bo_alpha)).fit(X_train, y_train)
bo_test_mse = mean_squared_error(y_test, bo_model.predict(X_test))

print(f"\nBayesian optimization: best alpha = {bo_alpha:.4g}"
      f"  (CV MSE {tried_mse[bo_best_idx]:.3f}, test MSE {bo_test_mse:.3f})"
      f"  in {len(tried_mse)} evaluations")
print(f"Grid search (2.3):     best alpha = {best_alpha:.4g}"
      f"  (CV MSE {-grid.best_score_:.3f}, test MSE {test_mse_best:.3f})"
      f"  in {len(alpha_grid)} evaluations")

In [ ]:
# Visualize what the optimizer "believes" at the end of the search.
# Refit the surrogate on ALL evaluated points (the loop above always fits
# before choosing, so the very last evaluation isn't in the GP yet).
gp.fit(np.array(tried_log_alpha).reshape(-1, 1), np.log10(tried_mse))
mu, sigma = gp.predict(candidates.reshape(-1, 1), return_std=True)

best_so_far = np.log10(min(tried_mse))
improvement = best_so_far - mu
z = np.where(sigma > 0, improvement / sigma, 0.0)
ei = improvement * norm.cdf(z) + sigma * norm.pdf(z)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), dpi=300, sharex=True)

ax1.plot(candidates, mu, 'b-', label='surrogate mean prediction')
ax1.fill_between(candidates, mu - 2*sigma, mu + 2*sigma, alpha=0.2,
                 label='uncertainty (±2 std)')
ax1.plot(tried_log_alpha, np.log10(tried_mse), 'ko', ms=6,
         label='evaluated points')
ax1.plot(tried_log_alpha[bo_best_idx], np.log10(tried_mse[bo_best_idx]),
         'r*', ms=18, label='best found')
ax1.set_ylabel('log10(CV MSE)')
ax1.set_title("The Gaussian-process surrogate after the final iteration")
ax1.legend(fontsize=9)

ax2.plot(candidates, ei, 'g-')
ax2.set_xlabel('log10(alpha)')
ax2.set_ylabel('Expected Improvement')
ax2.set_title('Acquisition function at the final iteration')

plt.tight_layout()
plt.show()

**Reading the result:**

- **Top panel**: the surrogate's belief about the CV error across the whole
  range. Near the black dots (evaluated points) the uncertainty band is pinched
  tight; far from them it balloons — the GP *knows what it doesn't know*.
- Notice **where the black dots cluster**: after a few spread-out probes, the
  optimizer spent its remaining budget around the valley. It may even re-try the
  incumbent best point — a sign the acquisition function sees nothing better
  left to explore, i.e. the search has **converged**.
- **Bottom panel**: the Expected Improvement is essentially zero everywhere at
  the end — no candidate is expected to beat the best found. Real libraries use
  exactly this as a stopping signal.
- **The scoreboard**: Bayesian optimization matched (here even slightly beat)
  the 30-point grid of section 2.3 using far fewer evaluations, because every
  try was *informed by all previous tries*.

For real problems — many hyperparameters, expensive training runs — use a
dedicated library such as **Optuna** or **scikit-optimize**; they wrap this exact
loop (better surrogates, smarter acquisition maximization, stopping rules) behind
a few lines of code.

## Summary

**Regularization**
- Overfitting = tiny training error but huge test error (a wiggly curve chasing noise).
- Regularization adds a penalty on coefficient size to the loss, favoring simpler
  models: *we regularize the model, not the data*.
- **L1 / Lasso** (Least Absolute Shrinkage and Selection Operator) can zero
  coefficients out — automatic **feature selection**, as the regularization path
  makes visible; with correlated features, *which* one survives is somewhat
  arbitrary. **L2 / Ridge** (named for the ridge it adds to the diagonal of
  $X^\top X$) shrinks all coefficients smoothly.
- The penalty strength `alpha` slides the model along the **bias–variance
  trade-off**: small `alpha` → high variance (overfit), large `alpha` → high bias
  (underfit). In classification the knob is `C` **(inverse: small `C` = strong
  penalty)**; in trees, limits like `max_depth` play the same role.

**Hyperparameter tuning**
- A **hyperparameter** is set *before* training (e.g. `alpha`, `C`, `max_depth`);
  a **learned parameter** is found *during* training (e.g. the coefficients).
- With only a train/test split you **cannot** tune — you'd be grading yourself on
  the final exam. **Cross-validation** creates fair validation scores from the
  training data alone.
- **`GridSearchCV`** automates trying combinations; a **validation curve** shows
  train vs. CV error so you can *see* the overfit/underfit (variance/bias) regimes.
- The **search range matters**: log-space scale-type knobs, and if the best value
  lands on the grid's edge, widen the grid.
- Grids grow multiplicatively with more hyperparameters (ElasticNet's
  `alpha` × `l1_ratio` was already 45 combos) — beyond a few, use **random
  search** or **Bayesian optimization** (surrogate model + acquisition function;
  Optuna / scikit-optimize in practice).

### Exercises for you — manual hyperparameter tuning

Everything `GridSearchCV` did for us is just a loop. Write it yourself once:
for each `alpha`, fit on the **training** points only, score on the held-out
**validation** points, and watch how the validation error moves. (We use a
simple hold-out split instead of k-fold CV here so every step is visible.)

In [ ]:
# --- Toy dataset (provided): a noisy sine wave, fit with degree-9 polynomial features ---
rng_ex = np.random.default_rng(7)
x_toy = np.linspace(-3, 3, 40)
y_toy = np.sin(1.5 * x_toy) + rng_ex.normal(0, 0.35, size=40)
X_toy = x_toy.reshape(-1, 1)

# Hold-out split: fit on the training part, score on the validation part.
X_tr, X_val, y_tr, y_val = train_test_split(X_toy, y_toy, test_size=0.3, random_state=0)

fig, ax = plt.subplots(figsize=(7, 4), dpi=300)
ax.plot(np.linspace(-3, 3, 300), np.sin(1.5 * np.linspace(-3, 3, 300)), 'g-', lw=1.5, label='true function')
ax.scatter(X_tr, y_tr, color='black', s=25, label='train')
ax.scatter(X_val, y_val, color='orange', s=35, marker='^', label='validation')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_title('Exercise 6 toy dataset')
ax.legend()
plt.show()

In [ ]:
# Worked example for ONE alpha -- your job: do this for every value in
# Tip: use make_pipeline(PolynomialFeatures(9), StandardScaler(), Ridge(alpha=a)) to create the model and use model.fit.